In [9]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [10]:
#LoadING the two datasets made by Apurba and Nishan
lr21 = pd.read_csv("LR_2_1_Apurba/data/linear_regression_2_1_candidate3_dataset.csv")
lr22 = pd.read_csv("linear_regression2.2/data/linear_regression_2_2_dataset.csv")

In [11]:
features21 = ["elo_difference", "squad_value_difference", "squad_experience_difference",
              "previous_possession_difference", "previous_goals_scored_difference",
              "defensive_advantage", "host_advantage", "previous_shots_on_target_difference"]
features22 = ["team_fifa_rank", "opponent_fifa_rank", "team_rest_days", "venue_elevation_meters",
              "team_prev_avg_corners", "team_prev_avg_fouls", "team_prev_avg_offsides", "referee_avg_cards"]

In [12]:
tasks = {
    "LR 2.1": (lr21, features21, "goal_difference"),
    "LR 2.2": (lr22, features22, "goals_scored"),
}

In [13]:
#the models we compare
def make_models():
    return {
        "Mean-only reference": DummyRegressor(strategy="mean"),                      # always predicts the average
        "OLS baseline": make_pipeline(StandardScaler(), LinearRegression()),         # the group's original model
        "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))),
        "Lasso": make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000)),
        "Huber (robust)": make_pipeline(StandardScaler(), HuberRegressor(max_iter=2000)),
    }

In [14]:
n_repeats = 10
n_folds = 5
 
results = []        # one row per task, model, repeat
lasso_kept = {}     # how often lasso keeps each variable

In [15]:
#repeated 5-fold cross-validation
for task_name, (data, features, target) in tasks.items():
    X = data[features].to_numpy()
    y = data[target].to_numpy()
    match_ids = data["match_id"].to_numpy()
    all_matches = np.unique(match_ids)
    kept_counts = np.zeros(len(features))
    kept_total = 0
 
    for repeat in range(n_repeats):
        # give each MATCH a fold number. In LR 2.2 both teams of a match get the same fold,
        # so the model never trains on one team and is tested on the other team of the same match.
        rng = np.random.default_rng(repeat)
        shuffled = rng.permutation(all_matches)
        fold_of_match = {m: i % n_folds for i, m in enumerate(shuffled)}
        fold = np.array([fold_of_match[m] for m in match_ids])
 
        predictions = {name: np.zeros(len(y)) for name in make_models()}
 
        for k in range(n_folds):
            train = fold != k
            test = fold == k
            models = make_models()
            for name, model in models.items():
                model.fit(X[train], y[train])
                predictions[name][test] = model.predict(X[test])
            # count which variables Lasso kept (coefficient not zero)
            kept_counts += (models["Lasso"][-1].coef_ != 0)
            kept_total += 1
 
        for name in predictions:
            results.append({
                "task": task_name, "model": name, "repeat": repeat,
                "MAE": mean_absolute_error(y, predictions[name]),
                "RMSE": np.sqrt(mean_squared_error(y, predictions[name])),
                "R2": r2_score(y, predictions[name]),
            })
 
    lasso_kept[task_name] = pd.Series(kept_counts / kept_total, index=features)
    print("finished", task_name)
 
results = pd.DataFrame(results)
results.to_csv("cv_repeat_level_results.csv", index=False)

finished LR 2.1
finished LR 2.2


In [16]:
#Average over the repeats
summary = results.groupby(["task", "model"]).agg(
    MAE=("MAE", "mean"), MAE_sd=("MAE", "std"),
    RMSE=("RMSE", "mean"), RMSE_sd=("RMSE", "std"),
    R2=("R2", "mean"), R2_sd=("R2", "std")).reset_index()
 
# check in what share of the repeats is each model better (lower RMSE) than the OLS baseline?
win_share = {}
for task_name in tasks:
    this_task = results[results["task"] == task_name]
    ols_rmse = this_task[this_task["model"] == "OLS baseline"].set_index("repeat")["RMSE"]
    for name in make_models():
        rmse = this_task[this_task["model"] == name].set_index("repeat")["RMSE"]
        win_share[(task_name, name)] = 100 * (rmse < ols_rmse).mean()
summary["better_than_OLS_in_%_repeats"] = [
    np.nan if row.model == "OLS baseline" else win_share[(row.task, row.model)]
    for row in summary.itertuples()]
summary = summary.round(3)
summary.to_csv("cv_model_comparison_full.csv", index=False)
 
# the short table for the report"mean (sd)"
table = pd.DataFrame({
    "task": summary["task"], "model": summary["model"],
    "MAE": summary["MAE"].astype(str) + " (" + summary["MAE_sd"].astype(str) + ")",
    "RMSE": summary["RMSE"].astype(str) + " (" + summary["RMSE_sd"].astype(str) + ")",
    "R2": summary["R2"].astype(str) + " (" + summary["R2_sd"].astype(str) + ")",
    "better_than_OLS_in_%_repeats": summary["better_than_OLS_in_%_repeats"]})
table.to_csv("cv_model_comparison_concise.csv", index=False)
 
pd.DataFrame(lasso_kept).round(2).rename_axis("variable").to_csv("lasso_selection_frequency.csv")

In [17]:
#Figure: RMSE of each model
order = ["Mean-only reference", "OLS baseline", "Ridge", "Lasso", "Huber (robust)"]
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.0))
for ax, task_name in zip(axes, tasks):
    s = summary[summary["task"] == task_name].set_index("model").loc[order]
    ax.barh(order, s["RMSE"], xerr=s["RMSE_sd"], color="steelblue", capsize=2.5)
    ax.axvline(s.loc["Mean-only reference", "RMSE"], color="black", linestyle="--", linewidth=0.8)
    ax.set_xlim(s["RMSE"].min() * 0.85, s["RMSE"].max() * 1.08)
    ax.invert_yaxis()
    ax.set_title(task_name, fontsize=9)
    ax.set_xlabel("Cross-validated RMSE (goals), mean +/- SD of repeats", fontsize=7.5)
    ax.tick_params(labelsize=7.5)
plt.tight_layout()
plt.savefig("cv_model_comparison.png", dpi=220)

In [18]:
print(table.to_string(index=False))
print()
print("How often Lasso kept each variable (1.00 = every fold):")
print(pd.DataFrame(lasso_kept).round(2))

  task               model           MAE          RMSE             R2  better_than_OLS_in_%_repeats
LR 2.1      Huber (robust) 1.304 (0.046) 1.656 (0.062)  0.323 (0.052)                          60.0
LR 2.1               Lasso 1.271 (0.044) 1.649 (0.065)  0.329 (0.054)                          40.0
LR 2.1 Mean-only reference 1.585 (0.007)  2.032 (0.01)  -0.018 (0.01)                           0.0
LR 2.1        OLS baseline 1.271 (0.048)   1.65 (0.07)  0.328 (0.059)                           NaN
LR 2.1               Ridge 1.257 (0.029) 1.625 (0.043)  0.349 (0.035)                         100.0
LR 2.2      Huber (robust) 0.999 (0.012)   1.3 (0.013)  0.126 (0.018)                           0.0
LR 2.2               Lasso 0.984 (0.009) 1.273 (0.013)  0.161 (0.018)                         100.0
LR 2.2 Mean-only reference 1.116 (0.003) 1.396 (0.005) -0.009 (0.007)                           0.0
LR 2.2        OLS baseline 0.999 (0.011) 1.291 (0.014)  0.137 (0.019)                           NaN
